# 04 · Shot detection with TransNet V2

Produces extra keyframes tagged `source=transnet`.

The organizers' I-frames are a *compression* artifact, not a semantic one: short
shots can fall between them entirely. TransNet V2 finds real shot boundaries, and
taking start/middle/end of each shot raises recall on brief events.

Frame numbers come straight from the model's boundary indices, so they are already
in the units we submit — no conversion, no rounding.

In [ ]:
# --- Colab setup -------------------------------------------------------------
# Runtime > Change runtime type > T4 GPU before running.
!nvidia-smi -L
!git clone -q https://github.com/YOUR_ORG/new_aic2026.git /content/aic || (cd /content/aic && git pull -q)
%cd /content/aic
!pip install -q pandas pyarrow pillow tqdm
import sys; sys.path.insert(0, "/content/aic/src")

In [ ]:
# --- Mount Drive -------------------------------------------------------------
# Keyframes go in, artifacts come out. Keeping both on Drive means an interrupted
# runtime resumes instead of restarting from zero.
from google.colab import drive

drive.mount('/content/drive')

from pathlib import Path

DATA = Path('/content/drive/MyDrive/aic2026')
KEYFRAMES = DATA / 'raw/keyframes'
DERIVED   = DATA / 'derived'
DERIVED.mkdir(parents=True, exist_ok=True)
print('keyframes:', KEYFRAMES, KEYFRAMES.exists())

In [ ]:
!pip install -q tensorflow ffmpeg-python
!git clone -q https://github.com/soCzech/TransNetV2.git /content/TransNetV2
import sys; sys.path.insert(0, '/content/TransNetV2/inference')

from transnetv2 import TransNetV2

transnet = TransNetV2(model_dir='/content/TransNetV2/inference/transnetv2-weights')

In [ ]:
import subprocess
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

VIDEOS = DATA / 'raw/videos'
OUT_FRAMES = DATA / 'raw/keyframes_transnet'; OUT_FRAMES.mkdir(parents=True, exist_ok=True)

def shot_frames(video_path):
    """Return (start, middle, end) frame numbers for every detected shot."""
    _, single, _ = transnet.predict_video(str(video_path))
    scenes = transnet.predictions_to_scenes(single)
    return [(int(s), int((s + e) // 2), int(e)) for s, e in scenes]

In [ ]:
rows = []
for video in tqdm(sorted(VIDEOS.glob('*.mp4'))):
    target = OUT_FRAMES / video.stem
    if target.exists():
        continue
    target.mkdir(parents=True, exist_ok=True)

    for shot_index, (start, middle, end) in enumerate(shot_frames(video)):
        for label, frame_no in (('s', start), ('m', middle), ('e', end)):
            name = f'{shot_index:05d}{label}_{frame_no}.jpg'
            subprocess.run(
                ['ffmpeg', '-v', 'error', '-i', str(video),
                 '-vf', f'select=eq(n\\,{frame_no})', '-vsync', '0',
                 '-frames:v', '1', str(target / name)], check=False)
            rows.append({'video_id': video.stem, 'frame_idx': frame_no,
                         'shot': shot_index, 'position': label})

pd.DataFrame(rows).to_parquet(DERIVED / 'transnet_frames.parquet', index=False)
print(f'{len(rows):,} keyframes from shot boundaries')

## Merge locally

Copy `keyframes_transnet/` into `data/raw/`, then rebuild the catalog including
both sources and re-run the SigLIP notebook over the enlarged catalog:

```bash
aic build-catalog --source transnet   # then merge with the btc_iframe catalog
```

Note the ordering constraint: **the catalog must be final before embeddings are
computed**, since `gid` is the embedding row index.